# Ejemplo de análisis espacial: altitud media por cantón

<div style="display: flex; justify-content: flex-start;">
  <a href="https://colab.research.google.com/github/tpb708-programacionsig/2026-i/blob/main/contenidos/iv-procesamiento-datos-geoespaciales/06-ejemplo-analisis-altitud-cantones.ipynb">
    <img src="https://img.shields.io/badge/Abrir%20en-Colab-F9AB00?logo=googlecolab&logoColor=white" alt="Abrir en Colab" style="margin: 0;">
  </a>
</div>

## Trabajo previo

### Lecturas

Tenkanen, H., Heikinheimo, V., & Whipp, D. (2024). *Introduction to Python for Geographic Data Analysis*. CRC Press. [https://pythongis.org/](https://pythongis.org/)
\
\
Lovelace, R., Nowosad, J., & Müller, J. (2024). *Geocomputation with Python*. CRC Press. [https://py.geocompx.org/](https://py.geocompx.org/)

### Otros recursos

Documentación oficial de rasterio\
[rasterio documentation](https://rasterio.readthedocs.io/)

  - [Masking a raster using a shapefile](https://rasterio.readthedocs.io/en/stable/topics/masking-by-shapefile.html)
  - [rasterio.mask — API reference](https://rasterio.readthedocs.io/en/stable/api/rasterio.mask.html)

Sistema Nacional de Información Territorial (SNIT) de Costa Rica\
[SNIT](https://www.snitcr.go.cr/)

WorldClim — datos climáticos globales\
[WorldClim](https://www.worldclim.org/)

Documentación oficial de GeoPandas\
[GeoPandas documentation](https://geopandas.org/)

Referencia de paletas de colores de matplotlib\
[Colormap reference — matplotlib documentation](https://matplotlib.org/stable/gallery/color/colormap_reference.html)

## Introducción

La **altitud media** de un polígono se define como el promedio de los valores de altitud de todos los píxeles de un raster que caen dentro de ese polígono:

$$
\text{altitud media} = \frac{1}{n} \sum_{i=1}^{n} z_i
$$

donde $z_i$ es la altitud de cada píxel y $n$ es el número de píxeles del raster contenidos en el polígono.

Este cuaderno de notas calcula la altitud media de cada cantón de Costa Rica a partir de la capa raster de altitud de [WorldClim](https://www.worldclim.org/) y la presenta en un mapa coroplético. Es un ejemplo típico de **estadísticas zonales**: el cálculo de un resumen (media, mínimo, máximo, etc.) de los valores de un raster dentro de cada uno de los polígonos de una capa vectorial.

## Carga de bibliotecas

In [1]:
# Carga de rasterio
import rasterio

# Módulo de rasterio para enmascarar un raster con polígonos
import rasterio.mask

# Carga de geopandas con el alias gpd
import geopandas as gpd

# Carga de pandas con el alias pd
import pandas as pd

# Biblioteca para álgebra lineal
import numpy as np

# Carga del módulo pyplot de matplotlib con el alias plt
import matplotlib.pyplot as plt

# Carga de la clase WebFeatureService del módulo wfs de owslib
from owslib.wfs import WebFeatureService

# Carga de la clase BytesIO del módulo estándar io
from io import BytesIO

# Carga de la biblioteca Folium, para mapas interactivos
import folium

## Carga de datos

### Altitud (WorldClim)

La capa raster de altitud para Costa Rica proviene del sitio [WorldClim](https://www.worldclim.org/) y está alojada en el repositorio del curso. Se abre con `rasterio.open()`.

In [2]:
# Lectura del raster de altitud (WorldClim)
altitud = rasterio.open(
    'https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/altitud.tif'
)

print(altitud)

<open DatasetReader name='https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/altitud.tif' mode='r'>


In [3]:
# Metadatos del raster
altitud.meta

{'driver': 'GTiff',
 'dtype': 'int16',
 'nodata': -32768.0,
 'width': 545,
 'height': 686,
 'count': 1,
 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'),
 'transform': Affine(0.008333333333333316, 0.0, -87.1,
        0.0, -0.008333333333333337, 11.216666666666669)}

### Cantones (WFS IGN 5k CO)

La capa vectorial de límites cantonales escala 1:5000 (CO) se obtiene del servicio WFS del Instituto Geográfico Nacional publicado en el SNIT. Primero listamos las capas disponibles en el servicio.

In [4]:
# Conexión al servicio WFS del IGN 5k CO
wfs_url = 'https://geos.snitcr.go.cr/be/IGN_5_CO/wfs'
wfs_version = '1.1.0'
wfs = WebFeatureService(url=wfs_url, version=wfs_version)

# Lista de las capas disponibles en el servicio WFS
for nombre_capa in wfs.contents:
    print(nombre_capa)

IGN_5_CO:limitedistrital_5k
IGN_5_CO:delimitacion2017_5k
IGN_5_CO:limitecantonal_5k
IGN_5_CO:limiteprovincial_5k
IGN_5_CO:linea_costa_5000


Luego se descarga la capa `IGN_5_CO:limitecantonal_5k` y se construye un `GeoDataFrame`.

In [5]:
# Obtener la capa de cantones
capa = 'IGN_5_CO:limitecantonal_5k'
respuesta = wfs.getfeature(typename=capa, outputFormat='application/json')

# Leer la respuesta en un GeoDataFrame
cantones_gdf = gpd.read_file(BytesIO(respuesta.read()))

# Reducción de columnas
cantones_gdf = cantones_gdf[['CÓDIGO_CANTÓN', 'CANTÓN', 'geometry']]

# Primeras filas
cantones_gdf.head()

,CÓDIGO_CANTÓN,CANTÓN,geometry
0,101,San José,"MULTIPOLYGON (((481030.328 1102633.167, 481032..."
1,102,Escazú,"MULTIPOLYGON (((479844.919 1102669.584, 479851..."
2,103,Desamparados,"MULTIPOLYGON (((494091.231 1094936.82, 494091...."
3,104,Puriscal,"MULTIPOLYGON (((455934.039 1095770.324, 455949..."
4,105,Tarrazú,"MULTIPOLYGON (((501998.931 1074558.189, 502000..."


El raster de WorldClim viene en EPSG:4326 (coordenadas geográficas WGS 84), mientras que la capa de cantones del SNIT viene en otro CRS (usualmente EPSG:5367 o EPSG:8908). Para calcular las estadísticas zonales es necesario que ambas capas estén en el mismo CRS. Aquí reproyectamos los **polígonos** al CRS del raster (EPSG:4326), porque reproyectar una capa vectorial es mucho más barato computacionalmente que reproyectar todos los píxeles del raster.

In [6]:
# Reproyectar los cantones al CRS del raster (EPSG:4326)
cantones_4326_gdf = cantones_gdf.to_crs(epsg=4326)

# Verificar el nuevo CRS
cantones_4326_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

## Análisis

### Cálculo de la altitud media por cantón

Para cada cantón se aplica `rasterio.mask.mask()` para extraer los píxeles del raster que caen dentro del polígono. El argumento `crop=True` recorta el resultado al *bounding box* del polígono y los píxeles fuera del polígono se rellenan con el valor `nodata` del raster. Luego se calcula la media con `numpy.nanmean()` ignorando los `NaN`.

In [7]:
# Valor de nodata del raster
nodata = altitud.nodata
print(f'Valor de nodata: {nodata}')

# Lista para acumular la altitud media de cada cantón
altitudes_medias = []

# Iterar sobre cada cantón
for _, fila in cantones_4326_gdf.iterrows():
    # Recortar el raster con el polígono del cantón
    recorte, _ = rasterio.mask.mask(
        altitud,
        [fila.geometry],
        crop=True
    )

    # Convertir a float y reemplazar nodata por NaN
    valores = recorte[0].astype('float32')
    valores = np.where(valores == nodata, np.nan, valores)

    # Calcular la media ignorando NaN
    altitudes_medias.append(np.nanmean(valores))

# Agregar la altitud media como una columna nueva al GeoDataFrame
cantones_4326_gdf['altitud_media'] = altitudes_medias

# Primeras filas
cantones_4326_gdf.head()

Valor de nodata: -32768.0


,CÓDIGO_CANTÓN,CANTÓN,geometry,altitud_media
0,101,San José,"MULTIPOLYGON (((-84.17302 9.97183, -84.173 9.9...",1099.924561
1,102,Escazú,"MULTIPOLYGON (((-84.18383 9.97215, -84.18377 9...",1311.949951
2,103,Desamparados,"MULTIPOLYGON (((-84.05388 9.90228, -84.05388 9...",1479.768066
3,104,Puriscal,"MULTIPOLYGON (((-84.40184 9.90958, -84.4017 9....",564.038635
4,105,Tarrazú,"MULTIPOLYGON (((-83.98178 9.71802, -83.98177 9...",1117.709229


In [8]:
# Diez cantones con mayor altitud media
cantones_4326_gdf[['CANTÓN', 'altitud_media']].sort_values(
    by='altitud_media',
    ascending=False
).head(10)

,CANTÓN,altitud_media
38,Alvarado,2082.031494
40,El Guarco,2067.575684
16,Dota,1952.433228
34,Paraíso,1884.872192
42,Barva,1851.590942
39,Oreamuno,1836.878662
33,Cartago,1805.631470
7,Goicoechea,1697.828613
19,León Cortés Castro,1676.216187
45,San Rafael,1634.118652


In [9]:
# Diez cantones con menor altitud media
cantones_4326_gdf[['CANTÓN', 'altitud_media']].sort_values(
    by='altitud_media',
    ascending=True
).head(10)

,CANTÓN,altitud_media
75,Los Chiles,56.421284
53,Carrillo,73.819695
72,Pococí,109.172318
64,Corredores,137.128372
51,Santa Cruz,144.119568
28,Orotina,146.718384
83,Puerto Jiménez,155.618484
65,Garabito,156.179138
61,Quepos,159.190552
50,Nicoya,168.369904


### Mapa coroplético

El método `explore()` de `GeoDataFrame` genera un mapa interactivo con Folium en el que cada cantón se colorea según su altitud media. Se usa la paleta `terrain` de matplotlib, que asigna tonos verdes a las zonas bajas y café/blanco a las altas — útil para representar elevaciones.

In [10]:
# Antes de graficar se simplifican los polígonos para reducir el
# tamaño del HTML resultante (los cantones de Costa Rica tienen
# muchos vértices y el GeoJSON embebido en el mapa puede pesar
# decenas de MB). Como el GeoDataFrame está en EPSG:4326, el
# factor está en grados; 0.001° ≈ 100 m, una tolerancia razonable
# para mapas a escala de país.
factor_simplificacion = 0.001

cantones_simplificado_gdf = cantones_4326_gdf.copy()
cantones_simplificado_gdf['geometry'] = (
    cantones_simplificado_gdf['geometry'].simplify(
        factor_simplificacion,
        preserve_topology=True
    )
)

# Mapa coroplético interactivo de altitud media por cantón
mapa = cantones_simplificado_gdf.explore(
    column='altitud_media',
    cmap='terrain',
    legend=True,
    tooltip=['CANTÓN', 'altitud_media'],
    popup=True,
    tiles='CartoDB positron',
    legend_kwds={'caption': 'Altitud media (m)'},
    style_kwds={'weight': 0.5, 'fillOpacity': 0.7},
    name='Altitud media por cantón'
)

# Agregar control de capas
folium.LayerControl().add_to(mapa)

# Mostrar el mapa
mapa

## Ejercicios

1. Repita el análisis con la capa raster de precipitación anual ([https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/precipitacion.tif](https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/precipitacion.tif)) en lugar de altitud. Calcule la precipitación media por cantón y construya el mapa coroplético correspondiente.

2. Repita el análisis con la capa raster de temperatura promedio anual ([https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/temperatura.tif](https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/temperatura.tif)) por **área de conservación** (capa WFS `PNE:areas_conservacion` del SINAC, alojada en `http://geos1pne.sirefor.go.cr/wfs`).